In [1]:
from neo4j import GraphDatabase
import pandas as pd
from dotenv import load_dotenv
import os

In [2]:
load_dotenv("../../BioPaster/.env")
PASSWD = os.getenv("NEO4J_PASSWORD")
BATCH_SIZE = 5000

### Install and set up Neo4j

In [ ]:
%%bash

docker compose \
  --env-file ../../BioPaster/.env \
  up -d
  
# # stop
# docker stop biopaster-neo4j
# # start
# docker start biopaster-neo4j

In [3]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=(
        "neo4j",
        PASSWD
    )
)


### Import nodes

#### Targets

In [10]:
def import_target(tx, rows):
    tx.run(
        """
        UNWIND $rows AS row

        MERGE (t:Target {id: row.target_id})

        SET
        t.symbol = row.symbol,
        t.name = row.name
        """,
        rows=rows
    )

In [ ]:
df = pd.read_csv("data/graph_data/nodes/targets.csv")

with driver.session() as session:
    for i in range(
        0,
        len(df),
        BATCH_SIZE
    ):
        batch = (
            df.iloc[i:i+BATCH_SIZE]
            .to_dict("records")
        )

        session.execute_write(
            import_target,
            batch
        )
        print(f"{i+BATCH_SIZE}/{len(df)}")


In [4]:
with driver.session() as session:
    result = session.run(
        "MATCH (n) RETURN count(n) AS count"
    )
    print(result.single()["count"])

148178


#### Diseases

In [22]:
def insert_diseases(tx, rows):
    tx.run(
        """
        UNWIND $rows AS row

        MERGE (d:Disease {id: row.disease_id})
        SET
            d.name = row.name
        """,
        rows=rows
    )

In [ ]:
df = pd.read_parquet("data/graph_data/nodes/diseases.parquet")

with driver.session() as session:
    for i in range(
        0,
        len(df),
        BATCH_SIZE
    ):
        batch = (
            df.iloc[i:i+BATCH_SIZE]
            .to_dict("records")
        )

        session.execute_write(
            insert_diseases,
            batch
        )

        print(f"{i+BATCH_SIZE}/{len(df)}")

In [5]:
with driver.session() as session:
    result = session.run(
        """
        MATCH (d:Disease)
        RETURN count(d) AS n
        """
    )
    print(result.single()["n"])

47080


#### Drugs

In [ ]:
def insert_drugs(tx, rows):
    tx.run(
        """
        UNWIND $rows AS row

        MERGE (d:Drug {id: row.drug_id})
        SET
            d.name = row.name,
            d.drug_type = row.drug_type,
            d.clinical_stage = row.clinical_stage,
            d.description = row.description
        """,
        rows=rows
    )


In [ ]:
df = pd.read_parquet("data/graph_data/nodes/drugs.parquet")

with driver.session() as session:
    for i in range(
        0,
        len(df),
        BATCH_SIZE
    ):
        batch = (
            df.iloc[i:i+BATCH_SIZE]
            .to_dict("records")
        )

        session.execute_write(
            insert_drugs,
            batch
        )

        print(f"{i+BATCH_SIZE}/{len(df)}")

5000/22407
10000/22407
15000/22407
20000/22407
25000/22407


In [6]:
with driver.session() as session:
    result = session.run(
        """
        MATCH (d:Drug)
        RETURN count(d) AS n
        """
    )
    print(result.single()["n"])

22407


Create index

In [ ]:
with driver.session() as session:
    session.run("""
        CREATE INDEX target_id IF NOT EXISTS
        FOR (n:Target) ON (n.id)
    """).consume()

    session.run("""
        CREATE INDEX disease_id IF NOT EXISTS
        FOR (n:Disease) ON (n.id)
    """).consume()

    session.run("""
        CREATE INDEX drug_id IF NOT EXISTS
        FOR (n:Drug) ON (n.id)
    """).consume()

### Import relationships

#### Disease-Targets

In [9]:
def insert_disease_target_edges(tx, rows):
    tx.run(
        """
        UNWIND $rows AS row

        MATCH (d:Disease {id: row.disease_id})
        MATCH (t:Target {id: row.target_id})

        MERGE (d)-[r:ASSOCIATED_WITH]->(t)

        SET
            r.datasource = row.datasource,
            r.score = toFloat(row.score),
            r.evidence_count = toInteger(row.evidence_count)
        """,
        rows=rows
    )

In [10]:
df = pd.read_parquet("data/graph_data/edges/disease_target.parquet")
df = df.sample(
    n=20_000,
    random_state=42
)

In [11]:
df.to_parquet("data/graph_data/edges/disease_target_demo.parquet")

In [12]:
with driver.session() as session:
    for i in range(
        0,
        len(df),
        BATCH_SIZE  
    ):
        batch = (
            df.iloc[i:i+BATCH_SIZE]
            .to_dict("records")
        )

        session.execute_write(
            insert_disease_target_edges,
            batch
        )

        print(f"{i+BATCH_SIZE}/{len(df)}")

5000/20000
10000/20000
15000/20000
20000/20000


In [20]:
with driver.session() as session:
    result = session.run(
        """
        MATCH ()-[r:ASSOCIATED_WITH]->()
        RETURN count(r) AS n
        """
    )
    print(result.single()["n"])

20000


#### Drugs-Targets

In [16]:
def insert_drug_target_edges(tx, rows):
    tx.run(
        """
        UNWIND $rows AS row

        MATCH (d:Drug {id: row.drug_id})
        MATCH (t:Target {id: row.target_id})

        CREATE (d)-[:TARGETS {
            action_type: row.action_type,
            mechanism: row.mechanism
        }]->(t)
        """,
        rows=rows
    )

In [18]:
df = pd.read_parquet(
    "data/graph_data/edges/drug_target.parquet"
)

In [21]:
with driver.session() as session:
    for i in range(
        0,
        len(df),
        BATCH_SIZE  
    ):
        batch = (
            df.iloc[i:i+BATCH_SIZE]
            .to_dict("records")
        )

        session.execute_write(
            insert_drug_target_edges,
            batch
        )

        print(f"{i+BATCH_SIZE}/{len(df)}")

5000/16355
10000/16355
15000/16355
20000/16355


In [26]:
with driver.session() as session:
    result = session.run("""
        MATCH ()-[r:TARGETS]->()
        RETURN count(r) AS n
    """)
    print(result.single()["n"])

15404


#### Clinical evidence

In [7]:
def insert_clinical_evidence(tx, rows):
    tx.run("""
        UNWIND $rows AS row

        MERGE (e:ClinicalEvidence {id: row.evidence_id})
        SET
            e.report_id = row.clinical_report_id,
            e.stage = row.clinical_stage,
            e.score = row.score

        WITH row, e

        OPTIONAL MATCH (d:Disease {id: row.disease_id})
        OPTIONAL MATCH (t:Target {id: row.target_id})
        OPTIONAL MATCH (drug:Drug {id: row.drug_id})

        FOREACH (_ IN CASE WHEN d IS NULL THEN [] ELSE [1] END |
            MERGE (d)-[:HAS_CLINICAL_EVIDENCE]->(e)
        )

        FOREACH (_ IN CASE WHEN t IS NULL THEN [] ELSE [1] END |
            MERGE (t)-[:HAS_CLINICAL_EVIDENCE]->(e)
        )

        FOREACH (_ IN CASE WHEN drug IS NULL THEN [] ELSE [1] END |
            MERGE (drug)-[:HAS_CLINICAL_EVIDENCE]->(e)
        )
    """, rows=rows)

In [5]:
df = pd.read_parquet("data/graph_data/edges/clinical_evidence.parquet")
df = df.sample(
    n=50_000,
    random_state=42
)

In [8]:
with driver.session() as session:
    for i in range(
        0,
        len(df),
        BATCH_SIZE  
    ):
        batch = (
            df.iloc[i:i+BATCH_SIZE]
            .to_dict("records")
        )

        session.execute_write(
            insert_clinical_evidence,
            batch
        )

        print(f"{i+BATCH_SIZE}/{len(df)}")

5000/50000
10000/50000
15000/50000
20000/50000
25000/50000
30000/50000
35000/50000
40000/50000
45000/50000
50000/50000


In [9]:
with driver.session() as session:

    # nodes
    evidence_nodes = session.run("""
        MATCH (e:ClinicalEvidence)
        RETURN count(e) AS n
    """).single()["n"]

    # edges
    evidence_edges = session.run("""
        MATCH ()-[r:HAS_CLINICAL_EVIDENCE]->(e:ClinicalEvidence)
        RETURN count(r) AS n
    """).single()["n"]

print("ClinicalEvidence nodes:", evidence_nodes)
print("HAS_CLINICAL_EVIDENCE relationships:", evidence_edges)

ClinicalEvidence nodes: 50000
HAS_CLINICAL_EVIDENCE relationships: 150000


### Query test

In [10]:
def query_target_graph(
    driver,
    disease: str,
    top_k: int = 10,
):
    query = """
    MATCH (d:Disease)
    WHERE toLower(d.name) CONTAINS toLower($disease)

    MATCH (d)-[a:ASSOCIATED_WITH]->(t:Target)

    OPTIONAL MATCH (drug:Drug)-[:TARGETS]->(t)
    OPTIONAL MATCH (t)-[:HAS_CLINICAL_EVIDENCE]->(e:ClinicalEvidence)

    RETURN
        d.id AS disease_id,
        d.name AS disease_name,
        t.id AS target_id,
        t.symbol AS target_symbol,
        max(a.score) AS association_score,
        collect(DISTINCT a.datasource) AS datasources,
        collect(DISTINCT drug.name)[0..5] AS drugs,
        count(DISTINCT e) AS clinical_evidence_count

    ORDER BY association_score DESC
    LIMIT $top_k
    """

    with driver.session() as session:
        result = session.run(
            query,
            disease=disease,
            top_k=top_k,
        )

        return result.data()

In [11]:
results = query_target_graph(
    driver,
    disease="acute myeloid leukemia",
    top_k=10,
)

for x in results:
    print(x)

{'disease_id': 'MONDO_0018874', 'disease_name': 'acute myeloid leukemia', 'target_id': 'ENSG00000146143', 'target_symbol': 'PRIM2', 'association_score': 0.9641561241705809, 'datasources': ['clinical_precedence'], 'drugs': ['CLOFARABINE', 'FLUDARABINE PHOSPHATE', 'TROXACITABINE', 'GEMCITABINE HYDROCHLORIDE', 'CYTARABINE'], 'clinical_evidence_count': 341}
{'disease_id': 'MONDO_0018874', 'disease_name': 'acute myeloid leukemia', 'target_id': 'ENSG00000119772', 'target_symbol': 'DNMT3A', 'association_score': 0.8354775769363881, 'datasources': ['intogen'], 'drugs': ['AZACITIDINE', 'DECITABINE'], 'clinical_evidence_count': 91}
{'disease_id': 'MONDO_0018874', 'disease_name': 'acute myeloid leukemia', 'target_id': 'ENSG00000138081', 'target_symbol': 'FBXO11', 'association_score': 0.8031617446237983, 'datasources': ['europepmc'], 'drugs': [], 'clinical_evidence_count': 0}
{'disease_id': 'MONDO_0018874', 'disease_name': 'acute myeloid leukemia', 'target_id': 'ENSG00000221829', 'target_symbol': '